In [1]:
import json
from pathlib import Path

import pandas as pd
import torch

torch.set_num_threads(4)
ROOT = Path.cwd()
if (ROOT / "ML").is_dir():
    ROOT = ROOT / "ML"
WYNIKI = ROOT / "wyniki" / "ml"
WYNIKI.mkdir(parents=True, exist_ok=True)

## Weights & Biases w CoreWeave Forge

Projekt [field-under-pressure](https://forge.coreweave.com/wandb/hackathon-field-under-pressure/field-under-pressure) jest ustawiony na stałe. Przed każdym eksperymentem zmień tylko `STANOWISKO`. Wpisany tekst pojawi się w Forge jako nazwa eksperymentu.

Kom?rka wymusza ponowne logowanie do Forge. Pobierz klucz ze [strony autoryzacji Forge](https://forge.coreweave.com/wandb/authorize) i wklej go w polu logowania. W terminalu odpowiednikiem jest `wandb login --relogin --host https://api.forge.coreweave.com`.

Gdy `WANDB = True`, zapisuj metryki przez `run.log({"val_accuracy": val_accuracy})`, a po treningu wywołaj `run.finish()`.

In [3]:
STANOWISKO = "Model bazowy"  # Wpisz, co robisz w tym eksperymencie.

WANDB = True
ZESPOL = "hackathon-field-under-pressure"
PROJEKT = "field-under-pressure"

if WANDB:
    import wandb

    if wandb.run is not None:
        raise RuntimeError("Zakończ aktywny eksperyment przed ponownym uruchomieniem tej komórki.")

    base_url = "https://api.forge.coreweave.com"

    if not wandb.login(host=base_url, relogin=True, verify=True):
        raise RuntimeError("Logowanie do Forge nie powiodło się. Sprawdź konfigurację W&B.")

    run = wandb.init(
        entity=ZESPOL,
        project=PROJEKT,
        name=STANOWISKO.strip(),
        job_type="train",
        mode="online",
        settings=wandb.Settings(base_url=base_url),
    )
    print(f"Eksperyment w Forge: {run.url}")
else:
    run = None

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.forge.coreweave.com. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://forge.coreweave.com/wandb/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Appending key for api.forge.coreweave.com to your netrc file: C:\Users\48881\_netrc


Eksperyment w Forge: https://forge.coreweave.com/wandb/hackathon-field-under-pressure/field-under-pressure/runs/9jcv8n69


**AISLOP**

In [ ]:
# Grid search dla klasyfikacji tabelarycznej. Przykład działa na danych syntetycznych.

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Podmień tę linię na X i y z projektu (cechy liczbowe i etykiety klas).
X, y = make_classification(n_samples=600, n_features=8, n_informative=5, n_classes=3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipeline = Pipeline([('scaler', 'passthrough'), ('model', LogisticRegression())])
param_grid = [
    {
        'scaler': [StandardScaler()],
        'model': [LogisticRegression(max_iter=1000)],
        'model__C': [0.1, 1.0, 10.0],
    },
    {
        'scaler': ['passthrough'],
        'model': [RandomForestClassifier(random_state=42)],
        'model__n_estimators': [50, 100],
        'model__max_depth': [5, None],
    },
]

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    n_jobs=1,
)
grid_search.fit(X_train, y_train)

najlepszy_model = grid_search.best_estimator_
f1_test = f1_score(y_test, najlepszy_model.predict(X_test), average='macro')

grid_wyniki = pd.DataFrame(grid_search.cv_results_)[
    ['params', 'rank_test_score', 'mean_test_score', 'std_test_score']
].rename(columns={
    'rank_test_score': 'ranking',
    'mean_test_score': 'f1_macro_cv',
    'std_test_score': 'f1_macro_cv_std',
})
grid_wyniki['model'] = grid_wyniki['params'].map(lambda params: type(params['model']).__name__)
grid_wyniki['parametry'] = grid_wyniki.pop('params').map(
    lambda params: json.dumps({key.removeprefix('model__'): value for key, value in params.items()
                               if key.startswith('model__')})
)
grid_wyniki = grid_wyniki.sort_values('ranking', ignore_index=True)

print(f'Najlepszy model: {grid_wyniki.loc[0, "model"]} ({grid_wyniki.loc[0, "parametry"]})')
print(f'F1 macro w walidacji: {grid_search.best_score_:.3f} | na zbiorze testowym: {f1_test:.3f}')

if globals().get('WANDB', False) and wandb.run is not None:
    wandb.run.log({
        'grid_search/f1_macro_cv': float(grid_search.best_score_),
        'grid_search/f1_macro_test': float(f1_test),
        'grid_search/wyniki': wandb.Table(dataframe=grid_wyniki),
    })

grid_wyniki